# Preprocessing - Wind Turbine Power Dataset

This notebook cleans the raw Kaggle wind turbine dataset and saves processed CSV files.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path('../datasets_raw/wind_turbine_power')
OUT_DIR = Path('../datasets')
OUT_DIR.mkdir(exist_ok=True)

In [2]:
train = pd.read_csv(RAW_DIR / 'Train.csv')
test = pd.read_csv(RAW_DIR / 'Test.csv')
column_info = pd.read_csv(RAW_DIR / 'column_info.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
display(train.head())
display(column_info)

Train shape: (140160, 12)
Test shape: (35040, 11)


,Unnamed: 0,Time,Location,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,Power
0,0,02-01-2013 00:00,1,28.2796,84.664205,24.072595,1.605389,1.267799,145.051683,161.057315,1.336515,0.163496
1,1,02-01-2013 01:00,1,28.1796,85.664205,24.272595,2.225389,3.997799,150.051683,157.057315,4.336515,0.142396
2,2,02-01-2013 02:00,1,26.5796,90.664205,24.072595,1.465389,2.787799,147.051683,149.057315,3.136515,0.121396
3,3,02-01-2013 03:00,1,27.1796,87.664205,23.872595,1.465389,2.697799,57.051683,104.057315,1.536515,0.100296
4,4,02-01-2013 04:00,1,27.0796,87.664205,23.672595,2.635389,4.437799,57.051683,83.057315,3.936515,0.079296


,Variables Description,Unnamed: 1
0,Time,Readings timestamp
1,Temp_2m,Temperature at 2 mtrs above surface
2,RelHum_2m,Relative Humidity at 2 mtrs above surface
3,DP_2m,Dew Point at 3 mtrs above surface
4,WS_10m,Wind Speed at 10 mtrs above surface
5,WS_100m,Wind Speed at 100 mtrs above surface
6,WD_10m,Wind Direction at 10 mtrs above surface
7,WD_100m,Wind Direction at 100 mtrs above surface
8,WG_10m,Wind Gusts at 100 mtrs above surface
9,Power,"Turbine power generation, normalized between 0..."


In [3]:
def clean_wind_dataset(df):
    df = df.copy()

    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    df.columns = [c.strip() for c in df.columns]
    df['Time'] = pd.to_datetime(df['Time'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['Time'])
    df = df.drop_duplicates()
    df = df.sort_values(['Location', 'Time']).reset_index(drop=True)

    for col in df.columns:
        if col != 'Time':
            df[col] = pd.to_numeric(df[col], errors='coerce')

    numeric_cols = [c for c in df.columns if c != 'Time']
    df[numeric_cols] = df.groupby('Location')[numeric_cols].transform(
        lambda s: s.interpolate(limit_direction='both')
    )

    df['year'] = df['Time'].dt.year
    df['month'] = df['Time'].dt.month
    df['day'] = df['Time'].dt.day
    df['hour'] = df['Time'].dt.hour
    df['date'] = df['Time'].dt.date.astype(str)

    df['wind_speed_difference_100m_10m'] = df['WS_100m'] - df['WS_10m']
    df['wind_speed_100m_squared'] = df['WS_100m'] ** 2
    df['wind_speed_100m_cubed'] = df['WS_100m'] ** 3

    df['wind_direction_100m_sin'] = np.sin(np.radians(df['WD_100m']))
    df['wind_direction_100m_cos'] = np.cos(np.radians(df['WD_100m']))
    df['wind_direction_10m_sin'] = np.sin(np.radians(df['WD_10m']))
    df['wind_direction_10m_cos'] = np.cos(np.radians(df['WD_10m']))

    return df

In [4]:
clean_train = clean_wind_dataset(train)
clean_test = clean_wind_dataset(test)

print(clean_train.shape)
print(clean_test.shape)
display(clean_train.head())
print(clean_train.isna().sum())

(140160, 23)
(35040, 22)


,Time,Location,Temp_2m,RelHum_2m,DP_2m,WS_10m,WS_100m,WD_10m,WD_100m,WG_10m,...,day,hour,date,wind_speed_difference_100m_10m,wind_speed_100m_squared,wind_speed_100m_cubed,wind_direction_100m_sin,wind_direction_100m_cos,wind_direction_10m_sin,wind_direction_10m_cos
0,2013-01-02 00:00:00,1,28.2796,84.664205,24.072595,1.605389,1.267799,145.051683,161.057315,1.336515,...,2,0,2013-01-02,-0.33759,1.607315,2.037753,0.324622,-0.945844,0.572837,-0.819669
1,2013-01-02 01:00:00,1,28.1796,85.664205,24.272595,2.225389,3.997799,150.051683,157.057315,4.336515,...,2,1,2013-01-02,1.77241,15.982400,63.894430,0.389810,-0.920895,0.499219,-0.866476
2,2013-01-02 02:00:00,1,26.5796,90.664205,24.072595,1.465389,2.787799,147.051683,149.057315,3.136515,...,2,2,2013-01-02,1.32241,7.771826,21.666291,0.514180,-0.857682,0.543882,-0.839162
3,2013-01-02 03:00:00,1,27.1796,87.664205,23.872595,1.465389,2.697799,57.051683,104.057315,1.536515,...,2,3,2013-01-02,1.23241,7.278122,19.634912,0.970053,-0.242892,0.839162,0.543882
4,2013-01-02 04:00:00,1,27.0796,87.664205,23.672595,2.635389,4.437799,57.051683,83.057315,3.936515,...,2,4,2013-01-02,1.80241,19.694064,87.398304,0.992668,0.120876,0.839162,0.543882


Time                              0
Location                          0
Temp_2m                           0
RelHum_2m                         0
DP_2m                             0
WS_10m                            0
WS_100m                           0
WD_10m                            0
WD_100m                           0
WG_10m                            0
Power                             0
year                              0
month                             0
day                               0
hour                              0
date                              0
wind_speed_difference_100m_10m    0
wind_speed_100m_squared           0
wind_speed_100m_cubed             0
wind_direction_100m_sin           0
wind_direction_100m_cos           0
wind_direction_10m_sin            0
wind_direction_10m_cos            0
dtype: int64


In [5]:
daily = clean_train.groupby(['Location', 'date'], as_index=False).agg(
    avg_power=('Power', 'mean'),
    max_power=('Power', 'max'),
    avg_wind_speed_100m=('WS_100m', 'mean'),
    avg_wind_speed_10m=('WS_10m', 'mean'),
    avg_temperature_2m=('Temp_2m', 'mean'),
    avg_relative_humidity_2m=('RelHum_2m', 'mean'),
    avg_wind_gust_10m=('WG_10m', 'mean')
)

monthly = clean_train.groupby(['Location', 'year', 'month'], as_index=False).agg(
    avg_power=('Power', 'mean'),
    avg_wind_speed_100m=('WS_100m', 'mean'),
    avg_wind_speed_10m=('WS_10m', 'mean'),
    avg_temperature_2m=('Temp_2m', 'mean'),
    reading_count=('Power', 'size')
)

display(daily.head())
display(monthly.head())

,Location,date,avg_power,max_power,avg_wind_speed_100m,avg_wind_speed_10m,avg_temperature_2m,avg_relative_humidity_2m,avg_wind_gust_10m
0,1,2013-01-02,0.194146,0.316496,4.944883,2.800806,32.254600,85.705871,5.328182
1,1,2013-01-03,0.230354,0.590496,4.356133,2.578723,36.617100,98.497538,4.969848
2,1,2013-01-04,0.868029,0.928996,12.382383,7.844556,19.758767,62.872538,15.924015
3,1,2013-01-05,0.698038,0.798196,8.505716,5.063723,10.221267,57.789205,10.324015
4,1,2013-01-06,0.322767,0.485696,6.985716,3.595389,5.408767,62.372538,6.699015


,Location,year,month,avg_power,avg_wind_speed_100m,avg_wind_speed_10m,avg_temperature_2m,reading_count
0,1,2013,1,0.376499,6.493522,3.781903,26.850156,720
1,1,2013,2,0.506574,7.353365,4.365330,32.435701,672
2,1,2013,3,0.481268,7.175326,4.416975,31.903659,744
3,1,2013,4,0.375711,7.246855,4.361570,48.429045,720
4,1,2013,5,0.449747,6.571294,4.028077,53.830272,744


In [6]:
clean_train.to_csv(OUT_DIR / 'wind_turbine_train_clean.csv', index=False)
clean_test.to_csv(OUT_DIR / 'wind_turbine_test_clean.csv', index=False)
daily.to_csv(OUT_DIR / 'wind_turbine_daily_summary.csv', index=False)
monthly.to_csv(OUT_DIR / 'wind_turbine_monthly_summary.csv', index=False)

print('Saved processed files to datasets folder.')

Saved processed files to datasets folder.
